# 02 缺失值与数据质量分析

本 notebook 对 Scania APS 原始 train/test 数据进行 Day 2 基础数据质量分析。当前阶段只分析数据规模、字段类型、标签分布、缺失值、缺失模式和重复行，不进行缺失值填充、字段删除、特征工程、模型训练或阈值优化。

## 1. 导入依赖

Day 2 只使用 pandas、numpy 和 matplotlib 完成基础统计与图表输出。

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 2. 设置项目根目录和数据路径

原始数据位于 `data/raw/`。本 notebook 只读取原始文件，不修改、覆盖、删除或重命名原始数据。

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "aps_failure_training_set.csv"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "aps_failure_test_set.csv"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH, TEST_PATH

(WindowsPath('c:/Scania APS/data/raw/aps_failure_training_set.csv'),
 WindowsPath('c:/Scania APS/data/raw/aps_failure_test_set.csv'))

## 3. 读取 train/test 数据

原始 CSV 中的 `na` 表示缺失值，读取时必须使用 `na_values="na"`。本阶段不合并 train/test 后重新划分。

In [3]:
train_df = pd.read_csv(TRAIN_PATH, na_values="na")
test_df = pd.read_csv(TEST_PATH, na_values="na")

LABEL_COL = "class"
TARGET_MAPPING = {"neg": 0, "pos": 1}

## 4. 基础规模检查

检查 train/test 的行数、列数、特征数、标签列是否存在，以及两份数据字段是否一致。

In [4]:
dataset_overview = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train_df),
            "columns": train_df.shape[1],
            "feature_count": train_df.shape[1] - int(LABEL_COL in train_df.columns),
            "has_label_column": LABEL_COL in train_df.columns,
            "train_test_columns_match": list(train_df.columns) == list(test_df.columns),
        },
        {
            "dataset": "test",
            "rows": len(test_df),
            "columns": test_df.shape[1],
            "feature_count": test_df.shape[1] - int(LABEL_COL in test_df.columns),
            "has_label_column": LABEL_COL in test_df.columns,
            "train_test_columns_match": list(train_df.columns) == list(test_df.columns),
        },
    ]
)

dataset_overview.to_csv(TABLES_DIR / "dataset_overview.csv", index=False, encoding="utf-8-sig")
dataset_overview

,dataset,rows,columns,feature_count,has_label_column,train_test_columns_match
0,train,60000,171,170,True,True
1,test,16000,171,170,True,True


## 5. 字段类型检查

检查字段类型分布，并确认除 `class` 外是否存在非数值字段。

In [5]:
type_rows = []
for dataset_name, df in [("train", train_df), ("test", test_df)]:
    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    other_cols = [col for col in df.columns if col not in object_cols and col not in numeric_cols]
    non_numeric_except_class = [col for col in object_cols + other_cols if col != LABEL_COL]
    type_rows.append(
        {
            "dataset": dataset_name,
            "object_count": len(object_cols),
            "numeric_count": len(numeric_cols),
            "other_type_count": len(other_cols),
            "non_numeric_except_class_count": len(non_numeric_except_class),
            "non_numeric_except_class": ";".join(non_numeric_except_class),
        }
    )

column_type_summary = pd.DataFrame(type_rows)
column_type_summary.to_csv(TABLES_DIR / "column_type_summary.csv", index=False, encoding="utf-8-sig")
column_type_summary

,dataset,object_count,numeric_count,other_type_count,non_numeric_except_class_count,non_numeric_except_class
0,train,1,170,0,0,
1,test,1,170,0,0,


## 6. 标签分布分析

`pos` 表示 APS 系统相关故障，`neg` 表示非 APS 系统相关样本。这个项目属于类别极不平衡问题，后续不能以 accuracy 作为核心指标，应重点关注 recall、F2、PR-AUC 和 total cost。

In [6]:
label_rows = []
for dataset_name, df in [("train", train_df), ("test", test_df)]:
    counts = df[LABEL_COL].value_counts(dropna=False)
    for label in ["neg", "pos"]:
        count = int(counts.get(label, 0))
        label_rows.append(
            {
                "dataset": dataset_name,
                "class": label,
                "count": count,
                "ratio": count / len(df),
            }
        )

label_distribution = pd.DataFrame(label_rows)
label_distribution.to_csv(TABLES_DIR / "label_distribution.csv", index=False, encoding="utf-8-sig")
label_distribution

,dataset,class,count,ratio
0,train,neg,59000,0.983333
1,train,pos,1000,0.016667
2,test,neg,15625,0.976562
3,test,pos,375,0.023438


In [7]:
train_df = train_df.assign(target=train_df[LABEL_COL].map(TARGET_MAPPING))
test_df = test_df.assign(target=test_df[LABEL_COL].map(TARGET_MAPPING))

train_df[[LABEL_COL, "target"]].head()

,class,target
0,neg,0
1,neg,0
2,neg,0
3,neg,0
4,neg,0


## 7. 缺失值总体分析

分别计算 train/test 每列缺失数量和缺失率。Day 2 只观察缺失情况，不做填充或删列。

In [8]:
def missing_summary(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    missing_count = df.isna().sum()
    result = pd.DataFrame(
        {
            "feature_name": missing_count.index,
            "missing_count": missing_count.values.astype(int),
            "total_count": len(df),
            "missing_rate": missing_count.values / len(df),
            "dataset": dataset_name,
        }
    )
    return result.sort_values("missing_rate", ascending=False).reset_index(drop=True)


missing_train = missing_summary(train_df.drop(columns=["target"]), "train")
missing_test = missing_summary(test_df.drop(columns=["target"]), "test")

missing_train.to_csv(TABLES_DIR / "missing_summary_train.csv", index=False, encoding="utf-8-sig")
missing_test.to_csv(TABLES_DIR / "missing_summary_test.csv", index=False, encoding="utf-8-sig")

missing_train.head(10)

,feature_name,missing_count,total_count,missing_rate,dataset
0,br_000,49264,60000,0.821067,train
1,bq_000,48722,60000,0.812033,train
2,bp_000,47740,60000,0.795667,train
3,bo_000,46333,60000,0.772217,train
4,ab_000,46329,60000,0.772150,train
5,cr_000,46329,60000,0.772150,train
6,bn_000,44009,60000,0.733483,train
7,bm_000,39549,60000,0.659150,train
8,bl_000,27277,60000,0.454617,train
9,bk_000,23034,60000,0.383900,train


## 8. 缺失率 Top N 字段分析

查看 train 数据缺失率最高的字段，为后续数据质量判断做准备。

In [9]:
missing_train[missing_train["feature_name"] != LABEL_COL].head(30)

,feature_name,missing_count,total_count,missing_rate,dataset
0,br_000,49264,60000,0.821067,train
1,bq_000,48722,60000,0.812033,train
2,bp_000,47740,60000,0.795667,train
3,bo_000,46333,60000,0.772217,train
4,ab_000,46329,60000,0.772150,train
5,cr_000,46329,60000,0.772150,train
6,bn_000,44009,60000,0.733483,train
7,bm_000,39549,60000,0.659150,train
8,bl_000,27277,60000,0.454617,train
9,bk_000,23034,60000,0.383900,train


## 9. 缺失率分层统计

将字段按缺失率划分为 `0%`、`0-5%`、`5-20%`、`20-50%`、`50-80%`、`80-100%` 六档。

In [10]:
def missing_level(rate: float) -> str:
    if rate == 0:
        return "0%"
    if rate <= 0.05:
        return "0-5%"
    if rate <= 0.20:
        return "5-20%"
    if rate <= 0.50:
        return "20-50%"
    if rate <= 0.80:
        return "50-80%"
    return "80-100%"


level_order = ["0%", "0-5%", "5-20%", "20-50%", "50-80%", "80-100%"]
level_rows = []
for dataset_name, summary_df in [("train", missing_train), ("test", missing_test)]:
    temp = summary_df.copy()
    temp["missing_level"] = temp["missing_rate"].map(missing_level)
    counts = temp["missing_level"].value_counts().reindex(level_order, fill_value=0)
    for level, count in counts.items():
        level_rows.append({"dataset": dataset_name, "missing_level": level, "feature_count": int(count)})

missing_level_summary = pd.DataFrame(level_rows)
missing_level_summary.to_csv(TABLES_DIR / "missing_level_summary.csv", index=False, encoding="utf-8-sig")
missing_level_summary

,dataset,missing_level,feature_count
0,train,0%,2
1,train,0-5%,127
2,train,5-20%,18
3,train,20-50%,16
4,train,50-80%,6
5,train,80-100%,2
6,test,0%,2
7,test,0-5%,127
8,test,5-20%,18
9,test,20-50%,16


## 10. 全空字段和高缺失字段检查

识别缺失率大于等于 50%、80% 和等于 100% 的字段。这里只做识别，不删除字段。

In [11]:
high_missing_features = pd.concat([missing_train, missing_test], ignore_index=True)
high_missing_features = high_missing_features[high_missing_features["missing_rate"] >= 0.50].copy()
high_missing_features["is_missing_ge_50"] = high_missing_features["missing_rate"] >= 0.50
high_missing_features["is_missing_ge_80"] = high_missing_features["missing_rate"] >= 0.80
high_missing_features["is_missing_eq_100"] = high_missing_features["missing_rate"] == 1.0
high_missing_features = high_missing_features.sort_values(["dataset", "missing_rate"], ascending=[True, False])

high_missing_features.to_csv(TABLES_DIR / "high_missing_features.csv", index=False, encoding="utf-8-sig")
high_missing_features.head(20)

,feature_name,missing_count,total_count,missing_rate,dataset,is_missing_ge_50,is_missing_ge_80,is_missing_eq_100
171,br_000,13129,16000,0.820562,test,True,True,False
172,bq_000,12981,16000,0.811312,test,True,True,False
173,bp_000,12721,16000,0.795063,test,True,False,False
174,bo_000,12376,16000,0.773500,test,True,False,False
175,ab_000,12363,16000,0.772687,test,True,False,False
176,cr_000,12363,16000,0.772687,test,True,False,False
177,bn_000,11713,16000,0.732062,test,True,False,False
178,bm_000,10546,16000,0.659125,test,True,False,False
0,br_000,49264,60000,0.821067,train,True,True,False
1,bq_000,48722,60000,0.812033,train,True,True,False


## 11. train/test 缺失模式对比

对比相同字段在 train 和 test 中的缺失率差异，观察两份数据是否存在明显分布差异。

In [12]:
train_test_missing_compare = missing_train[["feature_name", "missing_rate"]].rename(
    columns={"missing_rate": "train_missing_rate"}
).merge(
    missing_test[["feature_name", "missing_rate"]].rename(columns={"missing_rate": "test_missing_rate"}),
    on="feature_name",
    how="inner",
)
train_test_missing_compare["missing_rate_diff_abs"] = (
    train_test_missing_compare["train_missing_rate"] - train_test_missing_compare["test_missing_rate"]
).abs()
train_test_missing_compare = train_test_missing_compare.sort_values(
    "missing_rate_diff_abs", ascending=False
).reset_index(drop=True)

train_test_missing_compare.to_csv(TABLES_DIR / "train_test_missing_compare.csv", index=False, encoding="utf-8-sig")
train_test_missing_compare.head(30)

,feature_name,train_missing_rate,test_missing_rate,missing_rate_diff_abs
0,cl_000,0.159217,0.153688,0.005529
1,ed_000,0.159217,0.153688,0.005529
2,ec_00,0.170650,0.165250,0.005400
3,cm_000,0.164617,0.161125,0.003492
4,bk_000,0.383900,0.380875,0.003025
5,bl_000,0.454617,0.451625,0.002992
6,ca_000,0.072600,0.075375,0.002775
7,ar_000,0.045383,0.047750,0.002367
8,dx_000,0.045383,0.047750,0.002367
9,de_000,0.045400,0.047750,0.002350


## 12. 正负样本缺失情况对比

在 train 数据中分别计算 `pos` 和 `neg` 样本的字段缺失率，观察正负类是否存在不同缺失模式。这里只做观察，不能直接断言缺失本身一定可以作为特征，后续可以进一步评估。

In [13]:
feature_cols = [col for col in train_df.columns if col not in [LABEL_COL, "target"]]
neg_missing = train_df.loc[train_df[LABEL_COL] == "neg", feature_cols].isna().mean()
pos_missing = train_df.loc[train_df[LABEL_COL] == "pos", feature_cols].isna().mean()

class_missing_compare = pd.DataFrame(
    {
        "feature_name": feature_cols,
        "neg_missing_rate": neg_missing.reindex(feature_cols).values,
        "pos_missing_rate": pos_missing.reindex(feature_cols).values,
    }
)
class_missing_compare["missing_rate_diff_abs"] = (
    class_missing_compare["neg_missing_rate"] - class_missing_compare["pos_missing_rate"]
).abs()
class_missing_compare = class_missing_compare.sort_values("missing_rate_diff_abs", ascending=False).reset_index(drop=True)

class_missing_compare.to_csv(TABLES_DIR / "class_missing_compare.csv", index=False, encoding="utf-8-sig")
class_missing_compare.head(30)

,feature_name,neg_missing_rate,pos_missing_rate,missing_rate_diff_abs
0,br_000,0.833458,0.090,0.743458
1,bq_000,0.824322,0.087,0.737322
2,bp_000,0.807746,0.083,0.724746
3,bo_000,0.783983,0.078,0.705983
4,bn_000,0.744661,0.074,0.670661
5,bm_000,0.669119,0.071,0.598119
6,ak_000,0.065559,0.532,0.466441
7,ca_000,0.064847,0.530,0.465153
8,di_000,0.059017,0.524,0.464983
9,dj_000,0.059034,0.524,0.464966


## 13. 重复值检查

检查 train/test 是否存在完全重复行。

In [14]:
duplicate_summary = pd.DataFrame(
    [
        {
            "dataset": "train",
            "total_rows": len(train_df),
            "duplicate_rows": int(train_df.drop(columns=["target"]).duplicated().sum()),
            "duplicate_rate": float(train_df.drop(columns=["target"]).duplicated().mean()),
        },
        {
            "dataset": "test",
            "total_rows": len(test_df),
            "duplicate_rows": int(test_df.drop(columns=["target"]).duplicated().sum()),
            "duplicate_rate": float(test_df.drop(columns=["target"]).duplicated().mean()),
        },
    ]
)

duplicate_summary.to_csv(TABLES_DIR / "duplicate_summary.csv", index=False, encoding="utf-8-sig")
duplicate_summary

,dataset,total_rows,duplicate_rows,duplicate_rate
0,train,60000,0,0.0
1,test,16000,0,0.0


## 14. 保存基础图表

图表保存到 `outputs/figures/`，用于后续报告整理。

In [15]:
def save_barh(df: pd.DataFrame, x_col: str, y_col: str, title: str, xlabel: str, ylabel: str, path: Path) -> None:
    plot_df = df.iloc[::-1].copy()
    fig_height = max(6, len(plot_df) * 0.28)
    plt.figure(figsize=(10, fig_height))
    plt.barh(plot_df[y_col], plot_df[x_col], color="#4C78A8")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


train_label_plot = label_distribution[label_distribution["dataset"] == "train"].copy()
plt.figure(figsize=(6, 4))
plt.bar(train_label_plot["class"], train_label_plot["count"], color=["#4C78A8", "#F58518"])
plt.title("Train 标签分布")
plt.xlabel("class")
plt.ylabel("样本数")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "day2_label_distribution.png", dpi=160)
plt.close()

save_barh(
    missing_train[missing_train["feature_name"] != LABEL_COL].head(30),
    "missing_rate",
    "feature_name",
    "Train 缺失率 Top 30 字段",
    "缺失率",
    "字段名",
    FIGURES_DIR / "day2_missing_rate_top30_train.png",
)

save_barh(
    train_test_missing_compare[train_test_missing_compare["feature_name"] != LABEL_COL].head(30),
    "missing_rate_diff_abs",
    "feature_name",
    "Train/Test 缺失率差异 Top 30",
    "缺失率绝对差异",
    "字段名",
    FIGURES_DIR / "day2_train_test_missing_diff_top30.png",
)

save_barh(
    class_missing_compare.head(30),
    "missing_rate_diff_abs",
    "feature_name",
    "Pos/Neg 缺失率差异 Top 30",
    "缺失率绝对差异",
    "字段名",
    FIGURES_DIR / "day2_class_missing_diff_top30.png",
)

## 15. Day 2 小结

- 已完成 train/test 数据规模、字段类型和标签分布检查。
- 已完成 train/test 缺失率、缺失分层和高缺失字段识别。
- 已完成 train/test 缺失模式对比，以及 train 内部 pos/neg 缺失模式对比。
- 已完成重复行检查。
- 当前阶段没有做缺失值填充、字段删除、特征工程、模型训练或阈值优化。